#### What is this notebook about?
This is an intro to using llama (3.2) to develop AI agents.
I'll begin with rudimentary prompts, wrap them in functions, structure the output, and allow them to access tools (weather APIs, calendar, expand to other apps like spreadsheets and mail).
I'm following along with [this guide](https://www.youtube.com/watch?v=bZzyPscbtI8), except I'm transferring the workflow to LLama (which is catching up with its API, at the time of reading they might have more accessible features).

In [1]:
#!py -3.12 -m pip install ollama #If your python installation is something specific,
#(3.9<= is needed for pydantic -- see below)
#!pip install ollama #if you're normal
import ollama

To start with, I'm setting up a couple functions, a 'generic_response' which prompts you to give two strings, and get the response back.
Second, for my own readability, I save that output (which is a mess of \n and asterisks) as an html file.

In [5]:
def generic_response(rules, prompt):
    messages = [
    {"role":"system","content":rules},
    {"role":"user","content":prompt}
    ]
    print("Gimme a second to think...")
    response = ollama.chat(model='llama3.2',messages=messages)
    print("response generated")
    return response

def write_to_html(response, output_file):
    # Creating an HTML file
    print("writing it out to an html file...")
    Func = open(output_file,"w") 
    response_html = response['message']['content'].replace("\n", "<br>") #llama's line breaks to html line break
    response_html = response_html.replace(" **", "<b>")
    response_html = response_html.replace("** ", "</b>")
    response_html = response_html.replace("**:", "</b>:")
    response_html = "<html><head><title>llama response</title></head><body>"+response_html+"</body></html>"
    # Adding input data to the HTML file 
    Func.write(response_html)
    # Saving the data into the HTML file 
    Func.close()
    print("html file was created")

The above works well, and you can change the strings below and run it to see the change in 'html text.htm', or uncomment 'print response'to see the messy output displayed.

In [6]:
rules = "As an alley cat that knows english" #context in which you're interacting -- rools for the AI, roles to be played, etc
prompt = "how would you like to spend your life?" #what you actually prompt the llm to do

response = generic_response(rules, prompt)
print response
#write_to_html(response, "F:/html text.htm")z

Gimme a second to think...
writing it out to an html file...
html file was created


The following shows that you can also use it to work with data files, other strings, and so on.

In [ ]:
import ollama

def summarize_and_plan(notes_text):
    messages = [
        {"role": "system", "content": "You're an assistant that summarizes research notes and generates actionable tasks."},
        {"role": "user", "content": f"Here are my notes:\n\n{notes_text}\n\nPlease summarize them and suggest a task list."}
    ]

    response = ollama.chat(
        model='llama3.2',
        messages=messages
    )

    return response['message']['content']


# Example notes (generated by GPT, amusingly):
my_notes = """
- Read about reinforcement learning and Q-learning.
- Found a paper on reward shaping in sparse environments.
- Need to revisit value iteration vs policy iteration.
- Want to try implementing a simple RL agent in Python this week.
"""

result = summarize_and_plan(my_notes)
print(result)


This is a failed attempt to use ChromaDB (code from gpt), that needs some work on my part. (Put aside for now, though).

In [42]:
import chromadb
from chromadb.utils import embedding_functions
import ollama

# Set up ChromaDB
chroma_client = chromadb.Client()
client = chromadb.Client()
collection = client.get_or_create_collection("note_memory")

# Use Ollama’s embedding function
embedding_fn = embedding_functions.OllamaEmbeddingFunction(url = "http://localhost:8888/",model_name="nomic-embed-text")
def store_note(note_text, note_id):
    embedding = embedding_fn([note_text])[0]  # Get embedding from Ollama
    collection.add(
        documents=[note_text],
        embeddings=[embedding],
        ids=[note_id]
    )

def summarize_with_memory(current_note):
    similar_notes = get_similar_notes(current_note)

    prompt = (
        "You are a helpful assistant. A user has written new research notes.\n\n"
        f"New notes:\n{current_note}\n\n"
        f"Relevant past notes:\n{''.join(similar_notes)}\n\n"
        "Summarize the new notes and suggest actionable tasks based on both."
    )

    response = ollama.chat(
        model='llama3',
        messages=[{"role": "user", "content": prompt}]
    )

    return response['message']['content']
# User input (simulate session 1)
note1 = "Learned about decision trees and entropy in machine learning. Need to understand Gini index better."
store_note(note1, "note-001")

# User input (session 2)
note2 = "Read about random forests and how they build on decision trees. Confused about feature bagging."
store_note(note2, "note-002")

# Session 3: Summarize with memory
new_note = "Now I'm exploring gradient boosting. Not sure how it compares to random forests."
summary = summarize_with_memory(new_note)
print(summary)


TypeError: 'type' object is not subscriptable

#### Structured output systems using pydantic.

Output systems. We can specify that the output comes out in the form of a system. For instance, 'Jack and Jill went to party on Friday' will be converted into event.name = 'party', event.date = 'Friday', event.participants = ['Jack','Jill'], if we specify that the first two are strings and the last is a list of strings. 

Note: LLama (native) has the option to integrate format into the cURL / call itself, but we'll use Pydantic instead 
*Note: Pydantic is a well established data validation package, which allows you to build models
these reusable models make sure your input data conforms to a certain structure, and prevents bad data from making it through the gates*.

Here's an example:

In [18]:
from pydantic import BaseModel, Field

class CalendarEvent(BaseModel):
    destination: str
    date: str
    participants: list[str]
        
response = ollama.chat(
  messages=[
    {
      'role': 'user',
      'content': 'Jack and Jill went to party on Friday.',
    }
  ],
  model='llama3.2',
  ,
)
event = CalendarEvent.model_validate_json(response.message.content)
print(event)
print(event.destination) #you can print 
event.model_dump()

destination='party' date='friday' participants=['jack', 'jill']
party


{'destination': 'party', 'date': 'friday', 'participants': ['jack', 'jill']}

#### Letting the AI access tools
Now, I am going to experiment with retrieval and tools. From LLama 3.1, it has become able to match GPT's ability to use tools that are made available to it. For instance, apks, function calls, etc. It can't run the functions, but it can provide suggestions or instructions to pick one of the tools.
Then, feeding that tool output back into the LLM, we can output the information.
Confused? Chill, let's work through it together.

In [38]:
#let's make a more complex generic prompter first
def Prompt(prompt, tools, system_prompt = "you are a helpful AI assistant",model = 'llama3.2'):
    response = ollama.chat(model=model,
    messages=[{"role":"system","content":system_prompt},{'role': 'user', 'content':prompt}],
    tools = tools,)
    return response

In [39]:
#Now, we'll use this API to get a function that finds the weather in a city
import requests
def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]
#get_weather('51.178889', '-1.826111') #(test)

In [64]:
#for this one, we'll use a weather app APK
tools=[{
      'type': 'function',
      'function': {
        'name': 'get_weather',
        'description': 'Get the current temperature (in C) and windspeed for a location',
        'parameters': {
          'type': 'object',
          'properties': {
            'latitude': {'type': 'number','description': 'the latitude of a place',},
            'longitude':{'type': 'number','description': 'the longitude of a place',},
          },
          'required': ['latitude','longitude'],}, },},]

system_prompt ="you are a helpful weather assistant"
prompt = 'What is the weather like in Paris?'
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt},
]
response = Prompt(prompt, tools,system_prompt)
print(response['message']['tool_calls'])
#response.model_dump()

[ToolCall(function=Function(name='get_weather', arguments={'latitude': 48.8567, 'longitude': 2.3522}))]


From the above LLM call, we see that the model chose to call the function (we didn't need to explicitly give it the toronto lat and long, the LLM knew it) and we know the arguments now.
So, from the tool_calls, we can extract the arguments (by name of the tool)

In [26]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)

In [65]:
#Now, for each time the tool is called, we append the message given as response and append the result of 'get_weather'
for tool_call in response.message.tool_calls:
    name = tool_call.function.name
    args = tool_call.function.arguments
    #messages.append(response.message)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "content": json.dumps(result)}#This is important, othersie the 'content' becomes an unreadabele dict, instead of str
    )
messages

[{'role': 'system', 'content': 'you are a helpful weather assistant'},
 {'role': 'user', 'content': 'What is the weather like in Paris?'},
 {'role': 'tool',
  'content': '{"time": "2025-05-01T06:15", "interval": 900, "temperature_2m": 16.2, "wind_speed_10m": 5.0}'}]

Having appended the outputs to the messages section, the LLM will get this context  when it's fed the expanded messages.

In [60]:
messages

[{'role': 'system', 'content': 'you are a helpful weather assistant'},
 {'role': 'user', 'content': 'What is the weather like in Paris?'},
 {'role': 'tool',
  'content': {'time': '2025-05-01T06:00',
   'interval': 900,
   'temperature_2m': 15.9,
   'wind_speed_10m': 4.5}}]

In [66]:
from pydantic import BaseModel, Field
class WeatherResponse(BaseModel):
    temperature: float = Field(
        description="The current temperature in celsius for the given location."
    )
    response: str = Field(
        description="A natural language response to the user's question."
    )
class WeatherReport(BaseModel):
    temperature: float
    wind: float
    descriptive_report: str
        
response = ollama.chat(
  messages=messages,
  model='llama3.2',
    format = WeatherResponse.model_json_schema(),
    tools = tools,
    
)
print(response.message)#.content)
#event = WeatherResponse.model_validate(response.message.content)
#print(event)


role='assistant' content='{\n"temperature": 16.2,\n"wind": 5.0\n,"descriptive_report": "Partly cloudy"\n}\n\n   \t\t\t   \t\t\t\t\t\t\t   \t' images=None tool_calls=None


Since we've provided it with the tool call info already, it does not use the tool this time around:

In [72]:
response.model_dump()

{'model': 'llama3.2',
 'created_at': '2025-05-01T06:22:33.2020057Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 1798214600,
 'load_duration': 40674000,
 'prompt_eval_count': 121,
 'prompt_eval_duration': 6851000,
 'eval_count': 54,
 'eval_duration': 1737111200,
 'message': {'role': 'assistant',
  'content': '{"name": "Current Weather in Paris", "time": "2025-05-01T06:15", "interval": 900, "temperature_2m": 16.2, "wind_speed_10m": 5.0}',
  'images': None,
  'tool_calls': None}}

Structured input with llama still has issues compared to GPT, clearly (and this is the best alternative I have come up with so far, there may be others).

##### One more example
To start with, I'm running one more tool and another format - specifically, I'm providing a tool for the LLM to read a yaml text file- to make sure I've got the ideas down properly.
If you're following along, try to code this yourself, without checking what's below the imports cell.

In [138]:
#Creating a dataset of FAQ questions and answers
json_knowledge_base = {
    "records": [
        {
            "id": 1,
            "question": "What is the return policy?",
            "answer": "Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days."
        },
        {
            "id": 2,
            "question": "Do you ship internationally?",
            "answer": "Yes, we ship to over 50 countries worldwide. International shipping typically takes 7-14 business days and costs vary by destination. Please note that customs fees may apply."
        },
        {
            "id": 3,
            "question": "What payment methods do you accept?",
            "answer": "We accept Visa, Mastercard, American Express, PayPal, and Apple Pay. All payments are processed securely through our encrypted payment system."
        }
    ]
}
# Serializing json
json_object = json.dumps(json_knowledge_base, indent=4)
 
# Writing to sample.json
with open("F:/kb.json", "w") as outfile:
    outfile.write(json_object)

In [73]:
def search_kb(question:str):
    with open("F:/kb.json","r") as f:
        return json.load(f)

In [ ]:
import ollama
from pydantic import BaseModel, Field
import json

In [106]:
system_prompt = "You are a helpful AI assistant that answers questions from the  knowledge base about an e-commerce store"
user_prompt = "What is the return policy?"
messages = [
    {'role':'system','content':system_prompt},
    {'role':'user','content':user_prompt}
]
tools =[{
      'type': 'function',
      'function': {
        'name': 'search_kb',
        'description': "Get the answers to the user's question from the knowledge base",
        'parameters': {
          'type': 'object',
          'properties': {
            'question': {'type': 'string','description': 'the question to be answered',},
          },
          'required': ['question'],}, "strict": True },},]

Note: when I changed around the description a little bit, the question that the model got was different, so be careful. After said tweaking, however the model identifies the right tool_call

In [107]:
answer = ollama.chat(messages = messages, model='llama3.2', tools = tools)
answer.message.tool_calls
for tool_call in answer.message.tool_calls:
    if tool_call.function.name=='search_kb':
        result = search_kb(tool_call.function.arguments)
        messages.append(
            {"role": "tool", "content": json.dumps(result)}#This is important, othersie the 'content' becomes an unreadabele dict, instead of str
        )
messages

[{'role': 'system',
  'content': 'You are a helpful AI assistant that answers questions from the  knowledge base about an e-commerce store'},
 {'role': 'user', 'content': 'What is the return policy?'},
 {'role': 'tool',
  'content': '{"records": [{"id": 1, "question": "What is the return policy?", "answer": "Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days."}, {"id": 2, "question": "Do you ship internationally?", "answer": "Yes, we ship to over 50 countries worldwide. International shipping typically takes 7-14 business days and costs vary by destination. Please note that customs fees may apply."}, {"id": 3, "question": "What payment methods do you accept?", "answer": "We accept Visa, Mastercard, American Express, PayPal, and Apple Pay. All payments are processed securely through our encrypted payment system."}]}'}]

In [114]:
#Now, once more, we give the model a format to respond and the total messages and try again.
class KBResponse(BaseModel):
    answer: str = Field(description = "The answer to the user's question, framed rudely")
# You can't directly plug a Pydantic model into ollama , unfortunately -- here's a workaround. 
#GPT insists that it's problematic though, so come back to it at some point
final_answer = ollama.chat(messages = messages, model='llama3.2', tools = tools, format = KBResponse.model_json_schema())
print(final_answer.message.content)
kb = KBResponse.model_validate_json(final_answer.message.content)
#print(kb)

{"answer": "Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days."}


You now have the toolkit you need to build pretty much any AI agent, and if you followed attentively, you know how to construct functions, tools and structures that you can apply to any question.
From now on, I'll work through some examples, and how the three tricks we learned above can be put together to create a ready product.
### Prompt chaining : Building a calendar agent

You often need to chain prompt, route them or parallelize them. Let's start with prompt chaining.

Prompt chaining is a sequence of api calls, each prompt works on the output of the previous one. 
You can put in checks between them, terminating the process if a faulty response is output at any stage.

In [124]:
import logging
# Set up logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

from datetime import datetime
from typing import Optional


In [119]:
# To start with, let's develop some output models
class EventExtraction(BaseModel):
    '''First LLM call - is it a calendar event?'''
    description: str = Field(description="Raw description of the event")
    '''is_calendar_event is a gate, allowing you to stop if the user asks some unrelated nonsense'''
    is_calendar_event: bool = Field("Whether this text description contains a calendar event")
    confidence_score: float = Field(description="Confidence score between 0 and 1")
    
class EventDetails(BaseModel):
    '''Second LLM call: Parsing the information from raw text'''
    name: str = Field(description="name of the event")
    data: str = Field(description= " date and time of the event. Use ISO 8601 for this")
    duration_minutes: int = Field(description="Expected duration of the event")
    participants: list[str] = Field(description="list of participants")

class EventConfirmation(BaseModel):
    """Third LLM call: Generate confirmation message"""
    confirmation_message: str = Field(
        description="Natural language confirmation message"
    )
    calendar_link: Optional[str] = Field(
        description="Generated calendar link if applicable"
    )

In [135]:
model = 'llama3.2'

#The following chunks need editing, my bad. Check the url in the first block for more information, or debug this yourself (it's not too difficult)
#sorry about the inconvenience s
def extract_event_info(user_input: str) -> EventExtraction:
    """First LLM call to determine if input is a calendar event"""
    logger.info("Starting event extraction analysis")
    logger.debug(f"Input text: {user_input}")

    today = datetime.now()
    date_context = f"Today is {today.strftime('%A, %B %d, %Y')}."

    completion = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": f"{date_context} Analyze if the text describes a calendar event.",
            },
            {"role": "user", "content": user_input},
        ],
        format = EventExtraction.model_json_schema(),
    )
    result = completion.message.content
    print(result)
    logger.info(
        f"Extraction complete - Is calendar event: {result.is_calendar_event}, Confidence: {result.confidence_score:.2f}"
    )
    return result
#kb = KBResponse.model_validate_json(result.message.content)

def parse_event_details(description: str) -> EventDetails:
    """Second LLM call to extract specific event details"""
    logger.info("Starting event details parsing")

    today = datetime.now()
    date_context = f"Today is {today.strftime('%A, %B %d, %Y')}."

    completion = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": f"{date_context} Extract detailed event information. When dates reference 'next Tuesday' or similar relative dates, use this current date as reference.",
            },
            {"role": "user", "content": description},
        ],
        format = EventDetails.model_json_schema(),
    )
    result = completion.message.content
    logger.info(
        f"Parsed event details - Name: {result.name}, Date: {result.date}, Duration: {result.duration_minutes}min"
    )
    logger.debug(f"Participants: {', '.join(result.participants)}")
    return result

def generate_confirmation(event_details: EventDetails) -> EventConfirmation:
    """Third LLM call to generate a confirmation message"""
    logger.info("Generating confirmation message")

    completion = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Generate a natural confirmation message for the event. Sign of with your name; Susie",
            },
            {"role": "user", "content": str(event_details.model_dump())},
        ],
        format=EventConfirmation.model_json_schema(),
    )
    result = completion.message.content
    logger.info("Confirmation message generated successfully")
    return result

In [122]:
def process_calendar_request(user_input: str) -> Optional[EventConfirmation]:
    """Main function implementing the prompt chain with gate check"""
    logger.info("Processing calendar request")
    logger.debug(f"Raw input: {user_input}")

    # First LLM call: Extract basic info
    initial_extraction = extract_event_info(user_input)

    # Gate check: Verify if it's a calendar event with sufficient confidence
    if (
        not initial_extraction.is_calendar_event
        or initial_extraction.confidence_score < 0.7
    ):
        logger.warning(
            f"Gate check failed - is_calendar_event: {initial_extraction.is_calendar_event}, confidence: {initial_extraction.confidence_score:.2f}"
        )
        return None

    logger.info("Gate check passed, proceeding with event processing")

    # Second LLM call: Get detailed event information
    event_details = parse_event_details(initial_extraction.description)

    # Third LLM call: Generate confirmation
    confirmation = generate_confirmation(event_details)

    logger.info("Calendar request processing completed successfully")
    return confirmation


In [136]:
user_input = "Let's schedule a 1h team meeting next Tuesday at 2pm with Alice and Bob to discuss the project roadmap."

result = process_calendar_request(user_input)
if result:
    print(f"Confirmation: {result.confirmation_message}")
    if result.calendar_link:
        print(f"Calendar Link: {result.calendar_link}")
else:
    print("This doesn't appear to be a calendar event request.")
    


2025-05-01 13:38:14 - INFO - Processing calendar request
2025-05-01 13:38:14 - INFO - Starting event extraction analysis
2025-05-01 13:38:15 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


{"description": "Team Meeting", "confidence_score": 0.92, "is_calendar_event": true}


AttributeError: 'str' object has no attribute 'is_calendar_event'